# Previsione della domanda settimanale – Buby Rent

**Obiettivo:** prevedere il numero di noleggi effettivi (`stato_noleggio != 'annullata'`) per `(sede, tipo_veicolo, settimana_inizio)`, confrontare baseline e modelli ML su uno **split temporale rigoroso** (ultime 8 settimane complete tenute da parte) ed esportare le previsioni per le 4 settimane successive in `../../modelli/previsioni.csv`.

**Scelte metodologiche (`dai/README.md` §5 e `docs/PIANO_INTEGRAZIONE_DAI.md`):**
- **Settimane complete (lun–dom):** escludiamo le due settimane parziali ai bordi del dataset (`2025-10-27` e `2026-11-09`, solo 2 giorni ciascuna) per evitare cali artificiali.
- **Griglia completa:** 5 sedi × 3 tipi di veicolo × 53 settimane complete = **795 righe**, riempiendo con `0` le settimane senza noleggi (altrimenti i lag slitterebbero).
- **Metriche:** `MAE`, `RMSE`, `WAPE` (somma errori assoluti / somma noleggi reali, robusta agli zeri) e `MAPE` sulle righe con `y > 0`.
- **Nota per SAL 1 (27/11):** usiamo i noleggi effettivi (`conclusa` + `in_corso`, escludendo 430 annullati) e i nomi sede dell'app WSA (`Torino Lingotto`, `Alba`, `Bardonecchia`, `Stresa`, `Casale Monferrato`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
BASE_FILES = Path("../../files")
BASE_PULITI = Path("../../dati_puliti")
OUT_MODELLI = Path("../../modelli")
OUT_MODELLI.mkdir(parents=True, exist_ok=True)

nol_raw = pd.read_csv(BASE_PULITI / "noleggi_tutti_pulito.csv")
flotta = pd.read_csv(BASE_FILES / "flotta.csv")
sedi_df = pd.read_csv(BASE_FILES / "sedi.csv")

nome_sede_app = sedi_df.set_index("id_sede")["nome_sede"].str.replace(r"^Buby Rent ", "", regex=True).to_dict()

nol = nol_raw[nol_raw["stato_noleggio"] != "annullata"].copy()
nol["dt"] = pd.to_datetime(nol["data_ritiro"])
nol["settimana_inizio"] = nol["dt"] - pd.to_timedelta(nol["dt"].dt.weekday, unit="D")

# 53 settimane complete lun-dom: 2025-11-03 .. 2026-11-02 (ultima domenica: 2026-11-08)
PRIMA_SETT_COMPLETA = pd.Timestamp("2025-11-03")
ULTIMA_SETT_COMPLETA = pd.Timestamp("2026-11-02")
settimane = pd.date_range(PRIMA_SETT_COMPLETA, ULTIMA_SETT_COMPLETA, freq="W-MON")
SEDI = sorted(sedi_df["id_sede"].unique())
TIPI = ["auto", "ebike", "furgone"]

flotta["tipo_veicolo"] = flotta["tipo_mezzo"].replace({"e-bike": "ebike"})
cap = flotta.groupby(["id_sede", "tipo_veicolo"]).size().rename("n_veicoli").reset_index()
cap.columns = ["id_sede", "tipo_veicolo", "n_veicoli"]

griglia = pd.MultiIndex.from_product(
    [SEDI, TIPI, settimane], names=["id_sede", "tipo_veicolo", "settimana_inizio"]
).to_frame(index=False)

conteggi = (
    nol.groupby(["sede_ritiro", "tipologia_mezzo", "settimana_inizio"])
    .size()
    .rename("noleggi")
    .reset_index()
    .rename(columns={"sede_ritiro": "id_sede", "tipologia_mezzo": "tipo_veicolo"})
)

panel = (
    griglia.merge(conteggi, on=["id_sede", "tipo_veicolo", "settimana_inizio"], how="left")
    .fillna({"noleggi": 0})
    .merge(cap, on=["id_sede", "tipo_veicolo"], how="left")
)
panel["noleggi"] = panel["noleggi"].astype(int)

assert len(panel) == len(SEDI) * len(TIPI) * len(settimane) == 795
assert panel["n_veicoli"].notna().all()

n_esclusi_bordo = int(((nol["settimana_inizio"] < PRIMA_SETT_COMPLETA) | (nol["settimana_inizio"] > ULTIMA_SETT_COMPLETA)).sum())
print(
    f"Noleggi effettivi: {len(nol)} (esclusi {len(nol_raw) - len(nol)} annullati, "
    f"{n_esclusi_bordo} nelle 2 settimane parziali ai bordi)."
)
print(
    f"Pannello completo: {len(panel)} righe ({len(settimane)} settimane × 5 sedi × 3 tipi) | "
    f"media={panel['noleggi'].mean():.2f}, mediana={panel['noleggi'].median():.0f}, zeri={(panel['noleggi'] == 0).sum()}."
)
panel.groupby(["id_sede", "tipo_veicolo"])["noleggi"].agg(["mean", "min", "max"]).round(2)

Noleggi effettivi: 4567 (esclusi 430 annullati, 58 nelle 2 settimane parziali ai bordi).
Pannello completo: 795 righe (53 settimane × 5 sedi × 3 tipi) | media=5.67, mediana=5, zeri=65.


mean  min  max
id_sede tipo_veicolo                 
ALB     auto           6.74    2   11
        ebike          5.66    0   14
        furgone        2.28    0    7
BAR     auto           6.42    1   11
        ebike          4.08    0   13
        furgone        1.04    0    3
CAS     auto           5.68    3    9
        ebike          5.60    0   13
        furgone        2.51    0    5
STR     auto           6.43    1   12
        ebike          8.42    0   19
        furgone        0.81    0    3
TOR     auto          14.77   10   23
        ebike          9.32    2   20
        furgone        5.32    2    8

## 1. Feature di calendario, lag e split temporale

Tutte le feature storiche (`lag_1..4`, `roll_mean_4`, `roll_mean_8`) usano solo settimane **strettamente precedenti** (`shift >= 1`).
Le prime 4 settimane servono da warm-up per i lag:
- **Train:** `2025-12-01` → `2026-09-07` (**41 settimane**, 615 osservazioni).
- **Test (held-out):** `2026-09-14` → `2026-11-02` (**ultime 8 settimane complete**, 120 osservazioni).

In [2]:
# Festività nazionali italiane nel periodo 2025-11 .. 2026-12
# shortcut: static Italian national holiday list for 2025-2026; replace with `holidays` library if horizon extends beyond 2026
FESTIVI_IT = pd.to_datetime([
    "2025-11-01", "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-05", "2026-04-06",
    "2026-04-25", "2026-05-01", "2026-06-02", "2026-08-15",
    "2026-11-01", "2026-12-08", "2026-12-25", "2026-12-26",
])


def aggiungi_feature(df_in: pd.DataFrame) -> pd.DataFrame:
    p = df_in.sort_values(["id_sede", "tipo_veicolo", "settimana_inizio"]).copy()
    woy = p["settimana_inizio"].dt.isocalendar().week.astype(int)
    p["sin_woy"] = np.sin(2 * np.pi * woy / 52.0)
    p["cos_woy"] = np.cos(2 * np.pi * woy / 52.0)
    p["sin_woy2"] = np.sin(4 * np.pi * woy / 52.0)
    p["cos_woy2"] = np.cos(4 * np.pi * woy / 52.0)
    p["giorni_festivi"] = p["settimana_inizio"].apply(
        lambda w: int(((FESTIVI_IT >= w) & (FESTIVI_IT < w + pd.Timedelta(days=7))).sum())
    )
    g = p.groupby(["id_sede", "tipo_veicolo"])["noleggi"]
    for lag in (1, 2, 3, 4):
        p[f"lag_{lag}"] = g.shift(lag)
    p["roll_mean_4"] = g.transform(lambda s: s.shift(1).rolling(4).mean())
    p["roll_mean_8"] = g.transform(lambda s: s.shift(1).rolling(8, min_periods=4).mean())
    p["sede_tipo"] = p["id_sede"] + "_" + p["tipo_veicolo"]
    return p


panel_feat = aggiungi_feature(panel)
data = panel_feat.dropna(subset=["lag_4"]).copy()

CUTOFF_TEST = pd.Timestamp("2026-09-14")
train_df = data[data["settimana_inizio"] < CUTOFF_TEST].copy()
test_df = data[data["settimana_inizio"] >= CUTOFF_TEST].copy()

assert train_df["settimana_inizio"].max() < test_df["settimana_inizio"].min()
assert len(test_df) == 8 * len(SEDI) * len(TIPI) == 120

CAT_COLS = ["id_sede", "tipo_veicolo", "sede_tipo"]
NUM_COLS = [
    "n_veicoli", "sin_woy", "cos_woy", "sin_woy2", "cos_woy2",
    "giorni_festivi", "lag_1", "lag_2", "lag_3", "lag_4", "roll_mean_4", "roll_mean_8",
]
FEAT_COLS = CAT_COLS + NUM_COLS

print(f"Train: {len(train_df)} righe ({train_df['settimana_inizio'].min().date()} .. {train_df['settimana_inizio'].max().date()})")
print(f"Test : {len(test_df)} righe ({test_df['settimana_inizio'].min().date()} .. {test_df['settimana_inizio'].max().date()}), zeri nel test: {(test_df['noleggi'] == 0).sum()}")

Train: 615 righe (2025-12-01 .. 2026-09-07)
Test : 120 righe (2026-09-14 .. 2026-11-02), zeri nel test: 6


## 2. Confronto baseline e modelli ML sul `test` (ultime 8 settimane)

Valutiamo in due regimi realistici:
1. **1-step-ahead** (aggiornamento settimanale: ogni settimana i lag usano l'ultimo consuntivo reale).
2. **Orizzonte 4 settimane ricorsivo** (due blocchi consecutivi da 4 settimane sul `test`, senza leggere i consuntivi futuri: i lag delle settimane +2..+4 usano le previsioni precedenti).

Calcoliamo anche il **limite teorico di rumore di Poisson** (`E[|Y - λ|]` per conteggi intorno alla media del test) e le metriche aggregate **per sede** (`sede × settimana`), cioè la vista della dashboard.

In [3]:
pre_lin = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("num", StandardScaler(), NUM_COLS),
])
pre_tree = ColumnTransformer([
    ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), CAT_COLS),
    ("num", "passthrough", NUM_COLS),
])

MODELLI = {
    "Ridge": Pipeline([("pre", pre_lin), ("m", Ridge(alpha=30.0))]),
    "HGBR_poisson": Pipeline([
        ("pre", pre_tree),
        ("m", HistGradientBoostingRegressor(
            loss="poisson", max_depth=3, learning_rate=0.05,
            max_iter=150, min_samples_leaf=15, random_state=SEED,
        )),
    ]),
    "RandomForest": Pipeline([
        ("pre", pre_tree),
        ("m", RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=SEED)),
    ]),
}


def calcola_metriche(y_true, y_pred, nome: str, regime: str) -> dict:
    yt = np.asarray(y_true, dtype=float)
    yp = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    pos = yt > 0
    return {
        "regime": regime,
        "modello": nome,
        "MAE": round(float(mean_absolute_error(yt, yp)), 3),
        "RMSE": round(float(np.sqrt(np.mean((yt - yp) ** 2))), 3),
        "WAPE_%": round(float(np.abs(yt - yp).sum() / yt.sum() * 100), 1),
        "MAPE_pos_%": round(float(np.mean(np.abs((yt[pos] - yp[pos]) / yt[pos])) * 100), 1),
    }


def prevedi_ricorsivo(pipe: Pipeline, storico_base: pd.DataFrame, inizio: pd.Timestamp, orizzonte: int) -> pd.DataFrame:
    work = storico_base[storico_base["settimana_inizio"] < inizio][
        ["id_sede", "tipo_veicolo", "settimana_inizio", "noleggi", "n_veicoli"]
    ].copy()
    out = []
    for h in range(orizzonte):
        wk = inizio + pd.Timedelta(weeks=h)
        step = pd.MultiIndex.from_product(
            [SEDI, TIPI, [wk]], names=["id_sede", "tipo_veicolo", "settimana_inizio"]
        ).to_frame(index=False).merge(cap, on=["id_sede", "tipo_veicolo"], how="left")
        step["noleggi"] = np.nan
        comb = aggiungi_feature(pd.concat([work, step], ignore_index=True))
        cur = comb[comb["settimana_inizio"] == wk].copy()
        cur["noleggi"] = np.clip(pipe.predict(cur[FEAT_COLS]), 0.0, None)
        out.append(cur[["id_sede", "tipo_veicolo", "settimana_inizio", "noleggi"]])
        work = pd.concat([work, cur[["id_sede", "tipo_veicolo", "settimana_inizio", "noleggi", "n_veicoli"]]], ignore_index=True)
    return pd.concat(out, ignore_index=True)


# 1) Valutazione 1-step-ahead sul test (120 righe)
risultati = [
    calcola_metriche(test_df["noleggi"], test_df["lag_1"], "Baseline Naive (lag_1)", "1-step"),
    calcola_metriche(test_df["noleggi"], test_df["roll_mean_4"], "Baseline Media Mobile 4w", "1-step"),
]
pred_test_1step = {"Baseline_roll4": test_df["roll_mean_4"].values}

fitted = {}
for nome, pipe in MODELLI.items():
    pipe.fit(train_df[FEAT_COLS], train_df["noleggi"])
    fitted[nome] = pipe
    yp = np.clip(pipe.predict(test_df[FEAT_COLS]), 0.0, None)
    pred_test_1step[nome] = yp
    risultati.append(calcola_metriche(test_df["noleggi"], yp, nome, "1-step"))

# 2) Valutazione ricorsiva a 4 settimane (due blocchi da 4w dentro il test)
blocchi_4w = [CUTOFF_TEST, CUTOFF_TEST + pd.Timedelta(weeks=4)]

# Baseline statica su 4 settimane: media delle ultime 4 settimane note prima del blocco
yb_all, pb_all = [], []
for b in blocchi_4w:
    ult4 = panel[(panel["settimana_inizio"] < b) & (panel["settimana_inizio"] >= b - pd.Timedelta(weeks=4))]
    media4 = ult4.groupby(["id_sede", "tipo_veicolo"])["noleggi"].mean()
    sub = test_df[(test_df["settimana_inizio"] >= b) & (test_df["settimana_inizio"] < b + pd.Timedelta(weeks=4))]
    yb_all.append(sub["noleggi"].values)
    pb_all.append(sub.set_index(["id_sede", "tipo_veicolo"]).index.map(media4).values)
risultati.append(calcola_metriche(np.concatenate(yb_all), np.concatenate(pb_all), "Baseline Media Mobile 4w", "4w-ricorsivo"))

pred_test_4w = {}
for nome, pipe in fitted.items():
    fcs = [prevedi_ricorsivo(pipe, panel, b, 4) for b in blocchi_4w]
    fc_df = pd.concat(fcs, ignore_index=True).rename(columns={"noleggi": "pred"})
    m = test_df[["id_sede", "tipo_veicolo", "settimana_inizio", "noleggi"]].merge(
        fc_df, on=["id_sede", "tipo_veicolo", "settimana_inizio"], how="inner"
    )
    assert len(m) == len(test_df)
    pred_test_4w[nome] = m
    risultati.append(calcola_metriche(m["noleggi"], m["pred"], nome, "4w-ricorsivo"))

tab_risultati = pd.DataFrame(risultati)
display(tab_risultati)

# Rumore teorico di Poisson (riferimento sulla variabilità intrinseca di conteggi piccoli)
rng = np.random.default_rng(SEED)
lam = test_df["noleggi"].mean()
poisson_mae_floor = float(np.mean(np.abs(rng.poisson(lam, size=100_000) - lam)))
print(f"Riferimento rumore teorico Poisson (media test={lam:.2f} noleggi/sett): MAE atteso minimo ≈ {poisson_mae_floor:.2f}")

# Metriche aggregate per sede x settimana (vista dashboard CEO/operatore)
print("\n--- Aggregazione per sede × settimana (somma dei 3 tipi, regime 4w-ricorsivo) ---")
agg_rows = []
# Baseline per sede
base_4w_df = pd.concat([
    test_df[(test_df["settimana_inizio"] >= b) & (test_df["settimana_inizio"] < b + pd.Timedelta(weeks=4))][
        ["id_sede", "tipo_veicolo", "settimana_inizio", "noleggi"]
    ].assign(pred=lambda d, b=b: d.set_index(["id_sede", "tipo_veicolo"]).index.map(
        panel[(panel["settimana_inizio"] < b) & (panel["settimana_inizio"] >= b - pd.Timedelta(weeks=4))]
        .groupby(["id_sede", "tipo_veicolo"])["noleggi"].mean()
    ))
    for b in blocchi_4w
])
for nome, df_m in [("Baseline Media Mobile 4w", base_4w_df)] + list(pred_test_4w.items()):
    sede_agg = df_m.groupby(["id_sede", "settimana_inizio"])[["noleggi", "pred"]].sum()
    agg_rows.append(calcola_metriche(sede_agg["noleggi"], sede_agg["pred"], nome, "sede-4w"))
pd.DataFrame(agg_rows)

,regime,modello,MAE,RMSE,WAPE_%,MAPE_pos_%
0,1-step,Baseline Naive (lag_1),2.217,3.030,35.3,53.8
1,1-step,Baseline Media Mobile 4w,1.921,2.594,30.6,42.1
2,1-step,Ridge,1.825,2.460,29.1,40.4
3,1-step,HGBR_poisson,1.996,2.655,31.8,46.1
4,1-step,RandomForest,1.983,2.625,31.6,45.2
5,4w-ricorsivo,Baseline Media Mobile 4w,2.212,3.006,35.2,46.7
6,4w-ricorsivo,Ridge,1.943,2.640,30.9,41.7
7,4w-ricorsivo,HGBR_poisson,2.192,2.949,34.9,50.0
8,4w-ricorsivo,RandomForest,2.196,2.931,35.0,49.7


Riferimento rumore teorico Poisson (media test=6.28 noleggi/sett): MAE atteso minimo ≈ 2.00

--- Aggregazione per sede × settimana (somma dei 3 tipi, regime 4w-ricorsivo) ---


,regime,modello,MAE,RMSE,WAPE_%,MAPE_pos_%
0,sede-4w,Baseline Media Mobile 4w,4.175,5.101,22.1,31.3
1,sede-4w,Ridge,3.690,4.554,19.6,27.0
2,sede-4w,HGBR_poisson,4.317,5.204,22.9,33.9
3,sede-4w,RandomForest,4.436,5.243,23.5,35.0


## 3. Selezione modello, refit su tutto lo storico ed export `previsioni.csv`

Selezioniamo automaticamente il modello ML con `MAE` più basso nel regime **4w-ricorsivo** (il più aderente all'uso reale in produzione), salviamo il backtest sul `test` (`predizioni_domanda_test.csv`), ri-addestriamo il modello scelto su **tutte le 53 settimane complete** ed esportiamo le 4 settimane future (`2026-11-09` → `2026-11-30`) in `../../modelli/previsioni.csv` nel formato `docs/PIANO_INTEGRAZIONE_DAI.md` (`sede, tipo_veicolo, settimana_inizio, noleggi_previsti, modello_versione`).

In [4]:
# Scelta automatica del miglior modello ML sul regime 4w-ricorsivo
ml_4w = tab_risultati[(tab_risultati["regime"] == "4w-ricorsivo") & (tab_risultati["modello"].isin(MODELLI))]
best_name = ml_4w.sort_values("MAE").iloc[0]["modello"]
MODELLO_VERSIONE = f"{best_name.lower()}_v1"
print(f"Modello selezionato (miglior MAE 4w-ricorsivo): {best_name} -> versione '{MODELLO_VERSIONE}'")

# Salva backtest completo sul test (per report e grafici)
BACKTEST_OUT = OUT_MODELLI / "predizioni_domanda_test.csv"
bt = test_df[["id_sede", "tipo_veicolo", "settimana_inizio", "noleggi", "roll_mean_4"]].copy()
bt["sede"] = bt["id_sede"].map(nome_sede_app)
bt["pred_1step"] = np.round(pred_test_1step[best_name], 2)
bt = bt.merge(
    pred_test_4w[best_name][["id_sede", "tipo_veicolo", "settimana_inizio", "pred"]].rename(columns={"pred": "pred_4w_ricorsivo"}),
    on=["id_sede", "tipo_veicolo", "settimana_inizio"],
)
bt["pred_4w_ricorsivo"] = bt["pred_4w_ricorsivo"].round(2)
bt["settimana_inizio"] = bt["settimana_inizio"].dt.strftime("%Y-%m-%d")
bt["modello_versione"] = MODELLO_VERSIONE
bt.to_csv(BACKTEST_OUT, index=False)

# Refit del modello scelto su tutte le 49 settimane utilizzabili (53 complete meno 4 warm-up)
best_pipe = MODELLI[best_name]
best_pipe.fit(data[FEAT_COLS], data["noleggi"])

INIZIO_PREVISIONE = ULTIMA_SETT_COMPLETA + pd.Timedelta(weeks=1)  # 2026-11-09
ORIZZONTE_SETTIMANE = 4
fc_future = prevedi_ricorsivo(best_pipe, panel, INIZIO_PREVISIONE, ORIZZONTE_SETTIMANE)

previsioni = pd.DataFrame({
    "sede": fc_future["id_sede"].map(nome_sede_app),
    "tipo_veicolo": fc_future["tipo_veicolo"],
    "settimana_inizio": fc_future["settimana_inizio"].dt.strftime("%Y-%m-%d"),
    "noleggi_previsti": fc_future["noleggi"].round(1),
    "modello_versione": MODELLO_VERSIONE,
})

PREV_OUT = OUT_MODELLI / "previsioni.csv"
previsioni.to_csv(PREV_OUT, index=False)

# Verifiche finali di contratto
assert len(previsioni) == len(SEDI) * len(TIPI) * ORIZZONTE_SETTIMANE == 60
assert set(previsioni["sede"]) == set(nome_sede_app.values())
assert set(previsioni["tipo_veicolo"]) == {"auto", "furgone", "ebike"}
assert (previsioni["noleggi_previsti"] >= 0).all() and previsioni.notna().all().all()
assert BACKTEST_OUT.is_file() and PREV_OUT.is_file()

print(f"Salvato {BACKTEST_OUT} ({len(bt)} righe) e {PREV_OUT} ({len(previsioni)} righe).")
print("\n--- Domanda prevista aggregata per sede e settimana (prossime 4 settimane) ---")
previsioni.pivot_table(index="sede", columns="settimana_inizio", values="noleggi_previsti", aggfunc="sum").round(1)

Modello selezionato (miglior MAE 4w-ricorsivo): Ridge -> versione 'ridge_v1'


Salvato ..\..\modelli\predizioni_domanda_test.csv (120 righe) e ..\..\modelli\previsioni.csv (60 righe).

--- Domanda prevista aggregata per sede e settimana (prossime 4 settimane) ---


settimana_inizio,2026-11-09,2026-11-16,2026-11-23,2026-11-30
sede,,,,
Alba,14.8,13.6,12.2,12.0
Bardonecchia,8.4,8.1,7.1,6.8
Casale Monferrato,14.3,14.0,13.1,12.3
Stresa,13.1,13.5,11.8,11.2
Torino Lingotto,28.3,27.3,25.7,25.2
